# Ingest data from Bronze to Silver

In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F, types as T
from dotenv import load_dotenv
import os

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [3]:
# spark = (
#     SparkSession.builder.appName('Connect to Bronze via Trino')
#     .config("spark.jars", '../../../../spark/jar/trino-jdbc-480.jar') \
#     .getOrCreate()
# )
# spark

spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    # .config("spark.jars.packages", "io.delta:delta-spark_4.1_2.13:4.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # read postgres
    .config(
        'spark.jars',
        """
        ../../../../spark/jar/trino-jdbc-480.jar,
        ../../../../spark/jar/hadoop-aws-3.4.2.jar,
        ../../../../spark/jar/bundle-2.29.52.jar,
        ../../../../spark/jar/delta-spark_4.1_2.13-4.2.0.jar,
        ../../../../spark/jar/delta-kernel-api-4.2.0.jar,
        ../../../../spark/jar/delta-kernel-defaults-4.2.0.jar,
        ../../../../spark/jar/delta-storage-4.2.0.jar,
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", "s3a://flezi-data/warehouse/")
    .enableHiveSupport()
    .getOrCreate()
)

spark

26/06/22 06:28:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 06:28:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transfer

- Read from Trino

In [6]:
# Connection variables
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""
target_table = "datalake.taxi_data.nyc_taxi"

# Read table into a DataFrame
df = spark.read \
    .option("pushDownPredicate", "false") \
    .format("jdbc") \
    .option("url", trino_url) \
    .option("driver", "io.trino.jdbc.TrinoDriver") \
    .option("dbtable", target_table) \
    .option("user", trino_user) \
    .option("password", trino_password) \
    .load()

In [5]:
df.dtypes

[('vendor_id', 'string'),
 ('pickup_datetime', 'timestamp'),
 ('dropoff_datetime', 'timestamp'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('rate_code_id', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('pickup_longitude', 'double'),
 ('pickup_latitude', 'double'),
 ('dropoff_longitude', 'double'),
 ('dropoff_latitude', 'double'),
 ('pu_location_id', 'string'),
 ('do_location_id', 'string'),
 ('payment_type', 'string'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('surcharge', 'double'),
 ('improvement_surcharge', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double'),
 ('cbd_congestion_fee', 'double'),
 ('year', 'int'),
 ('month', 'int')]

In [7]:
df.filter(
    df['pickup_datetime'] > F.to_timestamp(F.lit('2009-01-01 00:00:00'))
).show()

26/06/22 06:30:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+----+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|year|month|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------

In [7]:
df.createOrReplaceTempView('nyc_taxi')
spark.sql("select count(*) from nyc_taxi where pickup_datetime > timestamp '2009-01-01 00:00:00' and pickup_datetime < timestamp '2009-01-02 00:00:00';").show()

26/06/22 03:44:37 ERROR Executor: Exception in task 0.0 in stage 3.0 (TID 2)
java.sql.SQLException: Query failed (#20260622_034437_00026_7qusb): line 4:63: Cannot apply operator: timestamp(3) < varchar(21)
	at io.trino.jdbc.ResultUtils.resultsException(ResultUtils.java:33)
	at io.trino.jdbc.TrinoResultSet.getColumns(TrinoResultSet.java:143)
	at io.trino.jdbc.TrinoResultSet.create(TrinoResultSet.java:49)
	at io.trino.jdbc.TrinoStatement.internalExecute(TrinoStatement.java:276)
	at io.trino.jdbc.TrinoStatement.execute(TrinoStatement.java:254)
	at io.trino.jdbc.TrinoPreparedStatement.executeQuery(TrinoPreparedStatement.java:149)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.$anonfun$compute$9(JDBCRDD.scala:336)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.withTimingNs(SQLMetrics.scala:234)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.compute(JDBCRDD.scala:335)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd

Py4JJavaError: An error occurred while calling o73.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3.0 (TID 2) (ip-10-0-0-6.ap-northeast-1.compute.internal executor driver): java.sql.SQLException: Query failed (#20260622_034437_00026_7qusb): line 4:63: Cannot apply operator: timestamp(3) < varchar(21)
	at io.trino.jdbc.ResultUtils.resultsException(ResultUtils.java:33)
	at io.trino.jdbc.TrinoResultSet.getColumns(TrinoResultSet.java:143)
	at io.trino.jdbc.TrinoResultSet.create(TrinoResultSet.java:49)
	at io.trino.jdbc.TrinoStatement.internalExecute(TrinoStatement.java:276)
	at io.trino.jdbc.TrinoStatement.execute(TrinoStatement.java:254)
	at io.trino.jdbc.TrinoPreparedStatement.executeQuery(TrinoPreparedStatement.java:149)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.$anonfun$compute$9(JDBCRDD.scala:336)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.with

- Cast some columns to int

In [4]:
cast_map = {
    'passenger_count': F.col('passenger_count').cast(T.IntegerType()),
    'pu_location_id' : F.col('pu_location_id').cast(T.IntegerType()),
    'do_location_id' : F.col('do_location_id').cast(T.IntegerType())
}
df = df.withColumns(cast_map)

In [5]:
def value_counts(df:DataFrame, col:str) -> DataFrame:
    return df.groupBy(col).count().orderBy('count')

- Transfer `payment_type`

In [6]:
# mapping
payment_map = {
    "0": "Flex Fare trip",

    # Cash
    'cash': 'Cash',
    "cas": "Cash",
    "csh": "Cash",
    "2": "Cash",

    # Credit card
    "credit": "Credit card",
    "crd": "Credit card",
    "cre": "Credit card",
    "1": "Credit card",

    # No charge
    "no charge": "No charge",
    "no": "No charge",
    "noc": "No charge",
    "3": "No charge",

    # Dispute
    "dispute": "Dispute",
    "dis": "Dispute",
    "4": "Dispute",

    # Unknown
    "na": "Unknown",
    "5": "Unknown",

    "6": "Voided trip",
}

# replace
df = (
    df
    .withColumn('payment_type', F.lower(F.trim(F.col('payment_type'))))
    .replace(payment_map, subset='payment_type')
)
value_counts(df, 'payment_type').show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|     Unknown|   1228|
|     Dispute|   2615|
|   No charge|  11208|
| Credit card|2798257|
|        Cash|4245924|
+------------+-------+



- Transfer `store_and_fwd_flag`

In [7]:
store_fwd_flag_map = {
    # yes
    '1': 'Yes',
    '1.0': 'Yes',
    'y': 'Yes',

    # no
    '0': 'No',
    '0.0': 'No',
    'n': 'No',

    # null
    'nan': None,
    '': None,
}

df = (
    df
    .withColumn('store_and_fwd_flag', F.lower(F.trim(F.col('store_and_fwd_flag'))))
    .replace(store_fwd_flag_map, subset='store_and_fwd_flag')
)
value_counts(df, 'store_and_fwd_flag').show()

+------------------+-------+
|store_and_fwd_flag|  count|
+------------------+-------+
|               Yes|  61405|
|                No|3311830|
|              NULL|3685997|
+------------------+-------+



In [8]:
tmp_df = value_counts(df, 'store_and_fwd_flag').toPandas()
len(tmp_df.iloc[:,0])

3

In [9]:
'Yes' in tmp_df.iloc[:,0].tolist()

True

- Transfer `vendor_id`

In [10]:
vendor_id_map = {
    'dds': 'DDS',

    "1": "Creative Mobile Technologies, LLC",
    'cmt': 'Creative Mobile Technologies, LLC',

    'vts': 'Curb Mobility, LLC',
    "2": "Curb Mobility, LLC",

    "6": "Myle Technologies Inc",

    "7": "Helix",
}

df = (
    df
    .withColumn('vendor_id', F.lower(F.trim(F.col('vendor_id'))))
    .replace(vendor_id_map, subset='vendor_id')
)
value_counts(df, 'vendor_id').show()

+--------------------+-------+
|           vendor_id|  count|
+--------------------+-------+
|                 DDS| 132159|
|  Curb Mobility, LLC|3457549|
|Creative Mobile T...|3469524|
+--------------------+-------+



## Load to LakeHouse

In [7]:
# spark.sql('drop schema taxi_data_silver cascade')
spark.sql('create database if not exists taxi_data_silver')

DataFrame[]

In [10]:
df_ = df.select('*').limit(1_000_000)
# df_.show()

In [11]:
S3_BRONZE_PATH = "s3a://flezi-data/silver/"
SCHEMA = "taxi_data_silver"
TABLE  = "nyc_taxi"

(
    df_
    .write
    .format('delta')
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

26/06/18 04:34:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/18 04:34:47 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`taxi_data_silver`.`nyc_taxi` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/06/18 04:34:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


Done. Table registered as 'taxi_data_silver.nyc_taxi' → s3a://flezi-data/silver/
